# AnyTime Universal Benchmark

Run early-exit benchmark for any AnyTime model. Switch by changing **one import line** below.

All HW metrics **per-PID** (process-scoped). Energy = `Î£ (power_w_i Ã— end_to_end_sec_i)` in Joules.

Per-run dir contains:
- `hw_results.json` â€” latency + memory + energy + per-PID GPU/CPU + FLOPs/params
- `quality_results.json` â€” accuracy / F1 / mAP / PPL (only if `skip_quality=False`)

Default sweep = HW-only. Pass `skip_quality=False` to enable quality eval.

Outputs:
- `logs/benchmark/{model}/...` â€” per-run JSONs (nested 2-3 levels deep)
- `results/{model}/` â€” CSV summaries


## 1. Setup

In [1]:
import numpy as np

In [2]:
import sys
import os
from pathlib import Path

REPO_ROOT = Path(".").resolve()
sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)

repo root: D:\Research\earlyexit\EarlyExitMonoRepo


In [3]:
# HF login (only needed if pulling from a private repo)
from shared import load_env

load_env()

try:
    from google.colab import userdata

    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN", ""))
except Exception:
    pass

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login

    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF: logged in")
else:
    print("HF: no token, public repos only")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF: logged in


## 2. Choose model

**Change ONE import line below** to swap which model gets benchmarked.

In [4]:
# # === SWAP THIS LINE TO PICK MODEL ===
# # from benchmark_config import bert as cfg
# from benchmark_config import llama  as cfg
# # from benchmark_config import vision as cfg
# # from benchmark_config import yolo   as cfg

# print("model :", cfg.NAME)
# print("family:", cfg.MODEL_FAMILY)
# print("out   :", cfg.OUT_DIR)

## 3. Run full sweep

Default = HW-only (`skip_quality=True`). Each run does:
- `profile_hw()` â€” HW + latency + per-PID power/VRAM/CPU + FLOPs/params + EDP
- `evaluate_quality()` â€” (skipped by default)

Override sweep dims via `only_*` kwargs (see `cfg.run_all` signature).


In [5]:
# cfg.run_all(skip_quality=False)          # HW + quality (all datasets)
# # cfg.run_all()                            # HW-only (cnn_dailymail latency only)
# # cfg.run_all(skip_quality=False, only_exit=4)  # single exit smoke test

## 3b. Run ALL models (full sweep)

In [ ]:
# # Run ALL models -- HW + quality sweep for every model family.
# # Comment out any model you want to skip.
# from benchmark_config import bert, llama, vision, yolo

# for _cfg in [bert, llama, vision, yolo]:
#     print(f"{60*chr(61)}")
#     print(f"  {_cfg.NAME} ({_cfg.MODEL_FAMILY})")
#     print(60*chr(61))
#     try:
#         _cfg.run_all(skip_quality=False,dry_run=True)  # HW + quality (all datasets)
#     except Exception as _e:
#         print(f"[{_cfg.NAME}] run_all failed: {_e}")
#         import traceback; traceback.print_exc()


In [ ]:
from shared import write_benchmark_csvs, average_across_tasks
from shared.averager import write_averaged_json
from benchmark_config import bert, llama, vision, yolo
from collections import defaultdict
import json
import pandas as pd

for _cfg in [bert, llama, vision, yolo]:
    out_dir = Path(_cfg.OUT_DIR)
    csv_root = REPO_ROOT / "results" / _cfg.NAME
    csv_root.mkdir(parents=True, exist_ok=True)

    hw_dirs = {p.parent for p in out_dir.rglob("hw_results.json")}
    q_dirs = {p.parent for p in out_dir.rglob("quality_results.json")}
    run_dirs = list(hw_dirs | q_dirs)
    if not run_dirs:
        print(f"[{_cfg.NAME}] no results found, skipping")
        continue
    print(f"[{_cfg.NAME}] found {len(run_dirs)} run dirs ({len(hw_dirs)} hw, {len(q_dirs)} quality)")

    # group by dataset parent dir; each group becomes one CSV bundle
    groups = defaultdict(dict)
    dataset_dirs = defaultdict(set)         # dataset_name -> {parent_dir_of_exits}
    for rd in run_dirs:
        rel_parent = rd.parent.relative_to(out_dir).as_posix()
        group_key = rel_parent.replace("/", "_") or "root"
        groups[group_key][rd.name] = rd
        dataset_dirs[group_key].add(rd.parent)

    # 1. per-dataset CSVs (one bundle per dataset)
    for group_key, runs in sorted(groups.items()):
        csv_dir = csv_root / group_key
        csv_dir.mkdir(parents=True, exist_ok=True)
        write_benchmark_csvs(
            results_files=runs,
            out_dir=csv_dir,
            baseline_key=None,
            method_order=sorted(runs.keys()),
        )
        print(f"  {group_key}: {len(runs)} runs -> {csv_dir}")

    # 2. cross-dataset averaged CSV (mean per method across all datasets)
    if len(groups) > 1:
        per_task = {k: next(iter(dirs)) for k, dirs in dataset_dirs.items()}
        averaged = average_across_tasks(per_task, run_pattern="*")
        if averaged:
            avg_root = csv_root / "_averaged"
            avg_root.mkdir(parents=True, exist_ok=True)
            # synthesize per-method hw_results.json so write_benchmark_csvs accepts it
            synth_runs = {}
            for method, metrics in averaged.items():
                md = avg_root / method
                md.mkdir(parents=True, exist_ok=True)
                (md / "hw_results.json").write_text(
                    json.dumps({"aggregate": metrics}, indent=2)
                )
                synth_runs[method] = md
            write_benchmark_csvs(
                results_files=synth_runs,
                out_dir=avg_root,
                baseline_key=None,
                method_order=sorted(synth_runs.keys()),
            )
            print(f"  _averaged: {len(averaged)} methods across {len(groups)} datasets -> {avg_root}")

print("All CSVs under:", REPO_ROOT / "results")


## 4b. Export CSVs — ALL models

## 4. Export CSVs

Recursive walk over per-run dirs (handles 2-3 level nesting per model). Merges
`hw_results.json` + `quality_results.json` if both exist. Groups by parent dir.


In [ ]:
from shared import write_benchmark_csvs
from collections import defaultdict

out_dir = Path(cfg.OUT_DIR)
csv_root = REPO_ROOT / "results" / cfg.NAME
csv_root.mkdir(parents=True, exist_ok=True)

hw_dirs = {p.parent for p in out_dir.rglob("hw_results.json")}
q_dirs = {p.parent for p in out_dir.rglob("quality_results.json")}
run_dirs = list(hw_dirs | q_dirs)
print(f"found {len(run_dirs)} run dirs under {out_dir} ({len(hw_dirs)} hw, {len(q_dirs)} quality)")

groups = defaultdict(dict)
for rd in run_dirs:
    rel_parent = rd.parent.relative_to(out_dir).as_posix()
    group_key = rel_parent.replace("/", "_") or "root"
    groups[group_key][rd.name] = rd

for group_key, runs in sorted(groups.items()):
    csv_dir = csv_root / group_key
    csv_dir.mkdir(parents=True, exist_ok=True)
    write_benchmark_csvs(
        results_files=runs,
        out_dir=csv_dir,
        baseline_key=None,
        method_order=sorted(runs.keys()),
    )
    print(f"  wrote {len(runs)} runs -> {csv_dir}")

print("CSVs under:", csv_root)


## 5. Quick view

In [9]:
import pandas as pd

for csv_name in ['latency.csv', 'energy.csv', 'quality.csv', 'hardware.csv']:
    for f in csv_root.rglob(csv_name):
        rel = f.relative_to(csv_root)
        print(f'=== {rel} ===')
        print(pd.read_csv(f).head(8))
        print()


In [ ]:
import re
import json
from pathlib import Path
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

# Per-backend mean+std plots, sourced from per-sample arrays in hw_results.json.
# Pools samples across all tasks/datasets per exit. Splits YOLO sub-exits
# (P3/P4/P5) into separate lines.
# Writes PNGs under results/{backend}/_plots/*.png

# (sample_key, pretty_title, y_label)
PANELS = [
    ("power_w",            "Power / Sample (W)",       "Power / Sample (W)"),
    ("energy_j",           "Energy / Sample (J)",      "Energy / Sample (J)"),
    ("end_to_end_sec",     "Latency e2e / Sample (s)", "Latency e2e / Sample (s)"),
    ("gpu_sm_clock_mhz",   "GPU Clock Speed (MHz)",    "GPU Clock Speed (MHz)"),
    ("vram_allocated_mb",  "GPU Used Memory (MB)",     "GPU Used Memory (MB)"),
    ("cpu_cores_used",     "CPU Cores Used",           "CPU Cores Used"),
    ("ram_used_mb",        "RAM Used (MB)",            "RAM Used (MB)"),
]

# exit_<int>[_<sub>]
_EXIT_RX = re.compile(r"^exit_(\d+)(?:_(.+))?$")

def _parse_exit(dirname):
    m = _EXIT_RX.match(dirname)
    if not m: return None
    return int(m.group(1)), (m.group(2) or "")

def _collect(backend_log_dir):
    """{(sub, main_exit): {sample_key: [values...]}, ...}, n_tasks."""
    pool = defaultdict(lambda: defaultdict(list))
    tasks = set()
    for hw in backend_log_dir.rglob("hw_results.json"):
        parsed = _parse_exit(hw.parent.name)
        if not parsed: continue
        main, sub = parsed
        tasks.add(hw.parent.parent.name)
        try:
            d = json.loads(hw.read_text())
        except Exception:
            continue
        for s in d.get("samples", []):
            for k, _t, _y in PANELS:
                v = s.get(k)
                if isinstance(v, (int, float)):
                    pool[(sub, main)][k].append(float(v))
    return pool, len(tasks)

def _plot_panel(pool, key, title, ylabel, n_tasks, out_path):
    # group by sub-exit -> ordered list of (main_exit, mean, std)
    by_sub = defaultdict(list)
    for (sub, main), kvs in pool.items():
        vals = kvs.get(key, [])
        if not vals: continue
        by_sub[sub].append((main, float(np.mean(vals)), float(np.std(vals, ddof=0))))
    if not by_sub:
        return False
    fig, ax = plt.subplots(figsize=(7, 4))
    colors = plt.cm.tab10.colors
    for i, sub in enumerate(sorted(by_sub.keys())):
        rows = sorted(by_sub[sub])
        x = np.array([r[0] + 1 for r in rows])           # 1-indexed exit layer
        m = np.array([r[1] for r in rows])
        s = np.array([r[2] for r in rows])
        label_mean = f"mean ({sub})" if sub else "mean"
        label_std  = f"\u00b1std ({sub}, n tasks={n_tasks})" if sub else f"\u00b1std (n tasks={n_tasks})"
        c = colors[i % len(colors)]
        ax.plot(x, m, marker="o", color=c, label=label_mean)
        ax.fill_between(x, m - s, m + s, color=c, alpha=0.20, label=label_std)
    ax.set_xlabel("Exit layer")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.show()
    plt.close(fig)
    return True

logs_root    = REPO_ROOT / "logs" / "benchmark"
results_root = REPO_ROOT / "results"
for backend_dir in sorted(p for p in logs_root.iterdir() if p.is_dir()):
    backend = backend_dir.name
    pool, n_tasks = _collect(backend_dir)
    if not pool:
        print(f"[{backend}] no hw_results.json, skip")
        continue
    plots_dir = results_root / backend / "_plots"
    plots_dir.mkdir(parents=True, exist_ok=True)
    print(f"=== {backend} (n_tasks={n_tasks}, exits={len(pool)}) ===")
    for key, title_suffix, ylabel in PANELS:
        title = f"{backend} \u2014 {title_suffix}"
        slug = re.sub(r"[^a-z0-9]+", "_", title_suffix.lower()).strip("_")
        out = plots_dir / f"{slug}.png"
        ok = _plot_panel(pool, key, title, ylabel, n_tasks, out)
        if ok: print(f"  -> {out}")
        else:  print(f"  skip {key} (no data)")

print("plots written under", results_root)
